# 04 · Train the models and create the submission

From the 10 external training matches (7 DFL d67s, 3 Metrica) we build three models that label the graded halves of Barcelona–Athletic:

| Step | Content |
|---|---|
| D model | Boosting: is the ball dead? Features after Lang et al. (2022) plus possession features |
| H model | Boosting: which team has the ball if it is alive? Features after Link & Hoernig (2017) plus robust team shape |
| Decoder | Viterbi over H/A/D with switching costs, then kick snap, kick-off rule and restart-team prior |

The switching costs are chosen by leave-one-match-out on the external matches. The challenge ground truth is not read.

**Output:** `output/HSA_Connection.csv` and `output/frozen_model.json`.

In [1]:
# Standard library
import hashlib
import json
import sys
import time
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Machine Learning
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve().parent
OFFICIAL_ZIP = ROOT / "reference" / "mtah2027_official_3de6b60.zip"
OFFICIAL = "more-than-a-hack-2027-3de6b60aeabd2475d95879a651d38b72bc4fba00"
sys.path.insert(0, str(OFFICIAL_ZIP / OFFICIAL / "evaluation"))
import possession_metrics as pm  # official scorer and validator, unchanged

TRAIN_DIR = ROOT / "artifacts" / "dfl_d67s"  # Notebook 02
MET_DIR = ROOT / "artifacts" / "metrica"  # Notebook 03
ATD_RAW = ROOT / "tactical" / "BAR-ATH_tactical_atd.txt"
ATD_META = ROOT / "tactical" / "BAR-ATH_tactical_atd.xml"
OUT = ROOT / "output"
OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION = OUT / "HSA_Connection.csv"

FPS = 25
N_FRAMES = 199_319
RANDOM_STATE = 42
PITCH_CH = np.array([105.3, 68.0])  # games.py
HALVES = [(11185, 90651), (110808, 190197)]  # DATA.md, inclusive
HALF_PITCH = (52.5, 34.0)  # DFL and Metrica; challenge uses PITCH_CH / 2
MAX_BALL_SPEED, MAX_PLAYER_SPEED = 40.0, 12.0
SUBSAMPLE = 5  # every 5th frame for training
GBM_PARAMS = dict(max_iter=300, learning_rate=0.1, max_leaf_nodes=31, l2_regularization=1.0, early_stopping=False, random_state=RANDOM_STATE)

# D features
CAP_S = 30.0
SHIFTS_S = [-40, -20, -4, -2, -0.8, 0, 0.8, 2, 4, 20, 40]
LANG_BASE = ["speed_med", "speed_q90", "moving", "acc_med", "spread", "centroid_absx", "centroid_dist", "ball_seen_s"]
# possession features
MARGIN_CLIP, NEAR_M = 15.0, 1.5
TAU_S, REACH_M, PRESS_M, REACH_CAP_S = 1.0, 2.0, 3.0, 5.0
LINK_SHIFTS_S = [-2, -1, -0.4, 0, 0.4, 1, 2]
TEAM_BASE_R = ["tilt_r", "push_r", "width_r", "depth_r", "adv_r"]
TEAM_SHIFTS_S = [-4, -2, -1, 0, 1, 2, 4]
# restarts
SLOW, QUIET_S, QUIET_SHARE, KICK_DT = 2.0, 1.0, 0.8, 5
KO_RADIUS, KO_DEPART, KO_DEPART_S, KO_LOOKBACK_S = 2.0, 4.0, 2.0, 120.0
SNAP = {"v_kick": 6.0, "w_before": 6.0, "w_after": 4.0}
RS_LOOK_S, RS_WINDOW_S, RS_NEAR_M = 5.0, 2.0, 5.0
RS_FEATURES = ["m_before", "m_after", "m_window", "n_diff"]
W_RS = 4  # weight of the restart prior; larger weights sat next to a performance drop during development
# Decoder
COST_GRID = [(c_ha, c_d) for c_ha in (5, 10, 20, 40) for c_d in (20, 40, 80, 160)]
VOTE_W = 50  # official baseline smooth50, comparison only

print(f"Python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · scikit-learn {sklearn.__version__}")

Python 3.11.7 · numpy 1.26.4 · pandas 2.3.3 · scikit-learn 1.6.1


## 1. Features

**Base and dead-ball context** (Lang et al. 2022): ball visibility and time since/until the ball was seen, distance of the last/next seen ball to the line, player speed, plus permutation-invariant team aggregates at 11 offsets from −40 to +40 s.

In [2]:
def clean(ball, players):
    ball, players = ball.copy(), players.copy()
    ball[np.r_[0, np.linalg.norm(np.diff(ball, axis=0), axis=1) * FPS] > MAX_BALL_SPEED] = np.nan
    step = np.linalg.norm(np.diff(players, axis=0), axis=-1) * FPS
    players[np.r_[np.zeros((1, players.shape[1])), step] > MAX_PLAYER_SPEED] = np.nan
    return ball, players

FEATURES = ["ball_seen", "seen_1s", "seen_5s", "seen_15s", "t_since_seen", "t_until_seen",
            "out_last", "out_next", "speed_1s", "speed_5s", "speed_15s", "moving_5s"]

def roll(x, seconds):
    return pd.Series(x).rolling(int(seconds * FPS) | 1, center=True, min_periods=1).mean().to_numpy()

def half_features(ball, players, half_pitch):
    n, idx = len(ball), np.arange(len(ball))
    seen = np.isfinite(ball).all(1)
    last = np.maximum.accumulate(np.where(seen, idx, -1))
    nxt = np.minimum.accumulate(np.where(seen, idx, n)[::-1])[::-1]

    def out_dist(j, valid):
        pos = ball[np.clip(j, 0, n - 1)]
        d = np.maximum(np.abs(pos[:, 0]) - half_pitch[0], np.abs(pos[:, 1]) - half_pitch[1])
        return np.where(valid, np.clip(d, -35, 10), np.nan)

    lag = 5  # ±0.2 s → difference over 0.4 s
    step = np.full(players.shape[:2], np.nan)
    step[lag:-lag] = np.linalg.norm(players[2 * lag:] - players[:-2 * lag], axis=-1) / (2 * lag / FPS)
    valid = np.isfinite(step)
    enough = valid.sum(1) >= 6
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # frames without valid players → NaN, intended
        speed = np.where(enough, np.nanmedian(np.where(valid, step, np.nan), axis=1), np.nan)
        moving = np.where(enough, (step > 2).sum(1) / valid.sum(1), np.nan)

    return pd.DataFrame({
        "ball_seen": seen.astype(float), "seen_1s": roll(seen, 1), "seen_5s": roll(seen, 5), "seen_15s": roll(seen, 15),
        "t_since_seen": np.where(last >= 0, np.minimum((idx - last) / FPS, CAP_S), CAP_S),
        "t_until_seen": np.where(nxt < n, np.minimum((nxt - idx) / FPS, CAP_S), CAP_S),
        "out_last": out_dist(last, last >= 0), "out_next": out_dist(nxt, nxt < n),
        "speed_1s": roll(speed, 1), "speed_5s": roll(speed, 5), "speed_15s": roll(speed, 15), "moving_5s": roll(moving, 5),
    })

def lang_half(ball, players, side):
    lag = 5
    vel = np.full(players.shape, np.nan)
    vel[lag:-lag] = (players[2 * lag:] - players[:-2 * lag]) / (2 * lag / FPS)
    speed = np.linalg.norm(vel, axis=-1)
    acc = np.full(speed.shape, np.nan)
    acc[lag:-lag] = np.linalg.norm(vel[2 * lag:] - vel[:-2 * lag], axis=-1) / (2 * lag / FPS)
    enough = np.isfinite(speed).sum(1) >= 6

    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        c_all = np.nanmean(players, axis=1)
        c_h, c_a = np.nanmean(players[:, side == "H"], axis=1), np.nanmean(players[:, side == "A"], axis=1)
        base = pd.DataFrame({
            "speed_med": np.nanmedian(speed, 1), "speed_q90": np.nanquantile(speed, 0.9, axis=1),
            "moving": (speed > 2).sum(1) / np.isfinite(speed).sum(1), "acc_med": np.nanmedian(acc, 1),
            "spread": np.nanmean(np.linalg.norm(players - c_all[:, None], axis=-1), 1),
            "centroid_absx": np.abs(c_all[:, 0]), "centroid_dist": np.linalg.norm(c_h - c_a, axis=1),
            "ball_seen_s": np.isfinite(ball).all(1).astype(float),
        })
    base.loc[~enough, LANG_BASE[:4]] = np.nan
    base = base.rolling(13, center=True, min_periods=1).mean()
    return pd.concat({f"{c}@{s:+g}": base[c].shift(-int(round(s * FPS))) for c in LANG_BASE for s in SHIFTS_S}, axis=1)

LANG_FEATURES = [f"{c}@{s:+g}" for c in LANG_BASE for s in SHIFTS_S]

**Possession** (Link & Hoernig 2017): how much closer the nearest H player is to the ball than the nearest A player, now and predicted 1 s ahead; how long each team has been within 2 m reach; players per team within 3 m; each with ±2 s context. Plus **robust team shape** (median and interquartile range of position, push, width and depth, ±4 s), because the challenge tracking contains outlier tracks. The playing direction is estimated from the deepest H player (usually the goalkeeper).

In [3]:
def run_length(mask):
    """Length (frames) of the True run containing each frame; 0 outside."""
    m = np.r_[False, mask, False].astype(int)
    e = np.flatnonzero(np.diff(m))
    out = np.zeros(len(mask))
    for a, b in zip(e[::2], e[1::2]):
        out[a:b] = b - a
    return out

def link_half(ball, players, side):
    lag, h = 5, 3
    vel = np.full(players.shape, np.nan)
    vel[lag:-lag] = (players[2 * lag:] - players[:-2 * lag]) / (2 * lag / FPS)
    vb = np.full(ball.shape, np.nan)
    vb[h:-h] = (ball[2 * h:] - ball[:-2 * h]) / (2 * h / FPS)

    def team_min(p, b):
        dist = np.linalg.norm(p - b[:, None], axis=-1)
        dist = np.where(np.isfinite(dist), dist, np.inf)
        d_h, d_a = dist[:, side == "H"].min(1), dist[:, side == "A"].min(1)
        ok = np.isfinite(d_h) & np.isfinite(d_a)
        return np.where(ok, d_h, np.nan), np.where(ok, d_a, np.nan), dist

    d_h, d_a, dist = team_min(players, ball)
    pd_h, pd_a, _ = team_min(players + np.nan_to_num(vel) * TAU_S, ball + vb * TAU_S)
    with np.errstate(invalid="ignore"):
        reach = (np.minimum(run_length(d_h <= REACH_M), REACH_CAP_S * FPS) - np.minimum(run_length(d_a <= REACH_M), REACH_CAP_S * FPS)) / FPS
        base = pd.DataFrame({
            "margin": np.clip(d_a - d_h, -MARGIN_CLIP, MARGIN_CLIP),
            "m_pred": np.clip(pd_a - pd_h, -MARGIN_CLIP, MARGIN_CLIP),
            "reach": np.where(np.isfinite(d_h), reach, np.nan),
        })
        extra = pd.DataFrame({
            "n3_diff": np.where(np.isfinite(d_h), (dist[:, side == "H"] < PRESS_M).sum(1) - (dist[:, side == "A"] < PRESS_M).sum(1), np.nan),
            "ball_speed": np.linalg.norm(vb, axis=1),
            "near_any": np.where(np.isfinite(d_h), (np.minimum(d_h, d_a) < NEAR_M).astype(float), np.nan),
        })
    shifted = pd.concat({f"{c}@{s:+g}": base[c].shift(-int(round(s * FPS))) for c in base for s in LINK_SHIFTS_S}, axis=1)
    return pd.concat([shifted, extra], axis=1)

def team_half(players, side):
    lag = 5
    vel = np.full(players.shape, np.nan)
    vel[lag:-lag] = (players[2 * lag:] - players[:-2 * lag]) / (2 * lag / FPS)
    H, A = side == "H", side == "A"
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        left, right = np.nanmedian(HALF_PITCH[0] + np.nanmin(players[:, H, 0], 1)), np.nanmedian(HALF_PITCH[0] - np.nanmax(players[:, H, 0], 1))
        dir_h = 1.0 if left < right else -1.0
        iqr = lambda q: np.nanquantile(q, 0.75, axis=1) - np.nanquantile(q, 0.25, axis=1)
        mx_h, mx_a = np.nanmedian(players[:, H, 0], 1), np.nanmedian(players[:, A, 0], 1)
        base = pd.DataFrame({
            "tilt_r": np.nanmedian(players[:, :, 0], 1) * dir_h,
            "push_r": (np.nanmedian(vel[:, H, 0], 1) - np.nanmedian(vel[:, A, 0], 1)) * dir_h,
            "width_r": iqr(players[:, H, 1]) - iqr(players[:, A, 1]),
            "depth_r": iqr(players[:, H, 0]) - iqr(players[:, A, 0]),
            "adv_r": (mx_h * dir_h + HALF_PITCH[0]) - (-mx_a * dir_h + HALF_PITCH[0]),
        }).rolling(13, center=True, min_periods=1).mean()
    return pd.concat({f"{c}@{s:+g}": base[c].shift(-int(round(s * FPS))) for c in TEAM_BASE_R for s in TEAM_SHIFTS_S}, axis=1), dir_h

LINK_FEATURES = [f"{c}@{s:+g}" for c in ["margin", "m_pred", "reach"] for s in LINK_SHIFTS_S] + ["n3_diff", "ball_speed", "near_any"]
TEAM_R_FEATURES = [f"{c}@{s:+g}" for c in TEAM_BASE_R for s in TEAM_SHIFTS_S]
H_FEATURES = LINK_FEATURES + TEAM_R_FEATURES
D_FEATURES = FEATURES + LANG_FEATURES + H_FEATURES  # the D model also sees the possession features

In [4]:
def per_half(fn, d, *keys):
    return [fn(*(d[k][d["period"] == p] if k != "side" else d[k] for k in keys)) for p in np.unique(d["period"])]

def margin(d):
    """Distance margin of H over A in m; basis of the restart model."""
    dist = np.linalg.norm(d["players"] - d["ball"][:, None], axis=-1)
    dist = np.where(np.isfinite(dist), dist, np.inf)
    d_h, d_a = (dist[:, d["side"] == s].min(1) for s in "HA")
    with np.errstate(invalid="ignore"):
        return np.where(np.isfinite(d_h) & np.isfinite(d_a), np.clip(d_a - d_h, -MARGIN_CLIP, MARGIN_CLIP), np.nan)

def features(d, half_pitch=HALF_PITCH):
    base = pd.concat(per_half(lambda b, p: half_features(b, p, half_pitch), d, "ball", "players"), ignore_index=True)
    lang = pd.concat(per_half(lang_half, d, "ball", "players", "side"), ignore_index=True).astype("float32")
    link = pd.concat(per_half(link_half, d, "ball", "players", "side"), ignore_index=True).astype("float32")
    team, d["directions"] = zip(*per_half(team_half, d, "players", "side"))
    d["F"] = pd.concat([base.astype("float32"), lang, link, pd.concat(team, ignore_index=True).astype("float32")], axis=1)
    d["margin"] = margin(d)
    return d

def load(path):
    df = pd.read_parquet(path)
    cols = [c[:-2] for c in df.columns if c.endswith("_x") and c[:2] in ("H_", "A_")]
    players = np.stack([df[[f"{c}_x" for c in cols]].to_numpy(), df[[f"{c}_y" for c in cols]].to_numpy()], axis=-1)
    return features({"y": (df["label"] == "D").to_numpy(), "label": df["label"].to_numpy(), "period": df["period"].to_numpy(),
                     "ball": df[["ball_x", "ball_y"]].to_numpy(), "players": players, "side": np.array([c[0] for c in cols])})

## 2. Models and decoder

- **D model** on all frames, **H model** on live frames only; both HistGradientBoosting with a fixed seed.
- **Viterbi** over H/A/D: an H↔A switch costs `c_HA`, a switch to D costs `c_D`. Short flickering therefore does not pay off.
- **Kick snap:** each end of a stoppage moves to the nearest ball onset above 6 m/s after a quiet phase (−6/+4 s).
- **Kick-off rule:** a ball that leaves the centre spot fast is a kick-off; the frames before it are `D`.
- **Restart team:** a logistic regression on the players around the ball predicts which team takes the restart. It enters a second decoder pass as extra cost (weight `W_RS`).

In [5]:
def fit_d(keys):
    X = pd.concat([data[k]["F"].iloc[::SUBSAMPLE] for k in keys])
    y = np.concatenate([data[k]["y"][::SUBSAMPLE] for k in keys])
    return HistGradientBoostingClassifier(**GBM_PARAMS).fit(X[D_FEATURES], y)

def fit_h(keys):
    X = pd.concat([data[k]["F"][~data[k]["y"]].iloc[::SUBSAMPLE] for k in keys])
    y = np.concatenate([(data[k]["label"] == "H")[~data[k]["y"]][::SUBSAMPLE] for k in keys])
    return HistGradientBoostingClassifier(**GBM_PARAMS).fit(X[H_FEATURES], y)

p_d = lambda model, d: model.predict_proba(d["F"][D_FEATURES])[:, 1]
p_h = lambda model, d: model.predict_proba(d["F"][H_FEATURES])[:, 1]

def viterbi3(lp, period, c_ha, c_d, c_dh=None, c_da=None):
    """lp: (n, 3) log emissions for H, A, D. c_dh/c_da: optional per-frame costs D→H/D→A."""
    n = len(lp)
    c_dh = np.full(n, float(c_d)) if c_dh is None else c_dh
    c_da = np.full(n, float(c_d)) if c_da is None else c_da
    out = np.empty(n, dtype="<U1")
    for h in np.unique(period):
        idx = np.flatnonzero(period == h)
        eh, ea, ed = (lp[idx, k].tolist() for k in range(3))
        dh, da = c_dh[idx].tolist(), c_da[idx].tolist()
        bp = np.zeros((len(idx), 3), np.int8)
        sh, sa, sd = eh[0], ea[0], ed[0]
        for t in range(1, len(idx)):
            nh, ph = sh, 0
            if sa - c_ha > nh: nh, ph = sa - c_ha, 1
            if sd - dh[t] > nh: nh, ph = sd - dh[t], 2
            na, pa = sa, 1
            if sh - c_ha > na: na, pa = sh - c_ha, 0
            if sd - da[t] > na: na, pa = sd - da[t], 2
            nd, pd_ = sd, 2
            if sh - c_d > nd: nd, pd_ = sh - c_d, 0
            if sa - c_d > nd: nd, pd_ = sa - c_d, 1
            bp[t] = ph, pa, pd_
            sh, sa, sd = nh + eh[t], na + ea[t], nd + ed[t]
        state = int(np.argmax([sh, sa, sd]))
        for t in range(len(idx) - 1, -1, -1):
            out[idx[t]] = "HAD"[state]
            state = bp[t, state]
    return out

def emissions(p_d, p_h):
    p_d, p_h = np.clip(p_d, 1e-6, 1 - 1e-6), np.clip(p_h, 1e-6, 1 - 1e-6)
    return np.column_stack([np.log1p(-p_d) + np.log(p_h), np.log1p(-p_d) + np.log1p(-p_h), np.log(p_d)])

In [6]:
def runs_of(dead, period):
    """[(start, stop, is_dead)] per half."""
    out = []
    for h in np.unique(period):
        idx = np.flatnonzero(period == h)
        x = dead[idx]
        cut = np.flatnonzero(np.r_[True, x[1:] != x[:-1], True])
        out += [(idx[0] + a, idx[0] + b, bool(x[a])) for a, b in zip(cut[:-1], cut[1:])]
    return out

def dead_spans(dead, period):
    """(start, first live frame) of every D spell that is followed by live play within the half."""
    r = runs_of(dead, period)
    return [(s0, s1) for (s0, e0, d0), (s1, e1, d1) in zip(r[:-1], r[1:]) if d0 and not d1 and e0 == s1]

def kick_events(ball, period, v_kick):
    sp = np.full(len(ball), np.nan)
    sp[:-KICK_DT] = np.linalg.norm(ball[KICK_DT:] - ball[:-KICK_DT], axis=1) / (KICK_DT / FPS)
    w = int(QUIET_S * FPS)
    quiet = pd.Series(~(sp > SLOW)).rolling(w, min_periods=w).mean().shift(1).to_numpy() >= QUIET_SHARE
    fast = sp > v_kick
    return np.flatnonzero(fast & ~np.r_[False, fast[:-1]] & quiet & np.r_[False, period[1:] == period[:-1]])

def snap_restarts(dead, period, kicks, v_kick, w_before, w_after):
    dead = dead.copy()
    r = runs_of(dead, period)
    for (s0, e0, d0), (s1, e1, d1) in zip(r[:-1], r[1:]):
        if not (d0 and not d1 and e0 == s1):
            continue
        lo, hi = max(s0 + 1, s1 - int(w_before * FPS)), min(e1 - 1, s1 + int(w_after * FPS))
        cand = kicks[(kicks >= lo) & (kicks <= hi)]
        if len(cand):
            k = cand[np.argmin(np.abs(cand - s1))]
            dead[min(s1, k):max(s1, k)] = k > s1
    return dead

def kickoff_rule(dead, period, ball, kicks):
    r = np.linalg.norm(ball, axis=1)
    far = pd.Series(r[::-1]).rolling(int(KO_DEPART_S * FPS), min_periods=1).max().to_numpy()[::-1]
    far = np.r_[far[1:], np.nan]
    dead = dead.copy()
    starts = np.array([s for s, e, d in runs_of(dead, period) if d])
    for k in kicks[(r[kicks] < KO_RADIUS) & (far[kicks] > KO_DEPART)]:
        prev = starts[(starts < k) & (starts >= k - KO_LOOKBACK_S * FPS) & (period[starts] == period[k])]
        if len(prev):
            dead[prev.max():k] = True
    return dead

def refine(lab, d):
    """Kick snap and kick-off rule; frames that become live get the team of the following (else previous) live spell."""
    k = kick_events(d["ball"], d["period"], SNAP["v_kick"])
    dead = kickoff_rule(snap_restarts(lab == "D", d["period"], k, **SNAP), d["period"], d["ball"], k)
    s = pd.Series(np.where(lab == "D", None, lab), dtype=object)
    s = s.groupby(d["period"]).transform(lambda x: x.bfill().ffill()).fillna("H").to_numpy()
    return np.where(dead, "D", np.where(lab == "D", s, lab))

In [7]:
def restart_features(d, starts, ends):
    m = d["margin"]
    seen = np.isfinite(m)
    look, win = int(RS_LOOK_S * FPS), int(RS_WINDOW_S * FPS)
    ball_ff = pd.DataFrame(d["ball"]).groupby(d["period"]).ffill().to_numpy()
    rows = []
    for s, e in zip(starts, ends):
        before = np.flatnonzero(seen[max(0, s - look):s])
        after = np.flatnonzero(seen[e:e + look])
        w = m[max(0, e - win):e + win]
        near = np.linalg.norm(d["players"][e] - ball_ff[e], axis=-1) < RS_NEAR_M
        rows.append({"m_before": m[max(0, s - look) + before[-1]] if len(before) else np.nan,
                     "m_after": m[e + after[0]] if len(after) else np.nan,
                     "m_window": np.nanmean(w) if np.isfinite(w).any() else np.nan,
                     "n_diff": float(near[d["side"] == "H"].sum() - near[d["side"] == "A"].sum()) if np.isfinite(ball_ff[e]).all() else np.nan})
    return pd.DataFrame(rows, columns=RS_FEATURES)

def fit_restart(keys):
    X, y = [], []
    for k in keys:
        sp = dead_spans(data[k]["y"], data[k]["period"])
        X.append(restart_features(data[k], *zip(*sp)))
        y.append(data[k]["label"][[e for _, e in sp]] == "H")
    model = make_pipeline(SimpleImputer(strategy="constant", fill_value=0.0, add_indicator=True), StandardScaler(),
                          LogisticRegression(max_iter=1000))
    return model.fit(pd.concat(X, ignore_index=True), np.concatenate(y))

def restart_costs(model, d, dead, c_d, w):
    c_dh, c_da = np.full(len(dead), float(c_d)), np.full(len(dead), float(c_d))
    sp = dead_spans(dead, d["period"])
    if sp:
        p = np.clip(model.predict_proba(restart_features(d, *zip(*sp)))[:, 1], 1e-3, 1 - 1e-3)
        win = int(RS_WINDOW_S * FPS)
        for (_, e), ph in zip(sp, p):
            sl = slice(max(0, e - win), e + win + 1)
            c_dh[sl], c_da[sl] = c_d - w * np.log(2 * ph), c_d - w * np.log(2 * (1 - ph))
    return c_dh, c_da

def decode(d, p_d, p_h, c_ha, c_d, rs_model, w=W_RS):
    lp = emissions(p_d, p_h)
    lab = refine(viterbi3(lp, d["period"], c_ha, c_d), d)
    return refine(viterbi3(lp, d["period"], c_ha, c_d, *restart_costs(rs_model, d, lab == "D", c_d, w)), d)

In [8]:
def score(d, labels):
    cut = np.flatnonzero(d["period"][1:] != d["period"][:-1]) + 1
    codes = lambda x: pm.label_codes(np.insert(np.asarray(x, dtype=object), cut, "X"))
    r = pm.evaluate(codes(d["label"]), codes(labels))
    return {"spell_f1": r["spell_f1_mean"], "macro_f1": r["macro_f1"], "alive_dead_f1": r["alive_dead_f1"],
            "n_pred_spells": r["n_pred_spells"], "n_gt_spells": r["n_gt_spells"]}

def baseline(d):
    """Official baseline smooth50: majority of the team closest to the ball over ±50 frames, D without a vote."""
    dist = np.linalg.norm(d["players"] - d["ball"][:, None], axis=-1)
    dist = np.where(np.isfinite(dist), dist, np.inf)
    team = np.where(np.isfinite(dist.min(1)), d["side"][dist.argmin(1)], "")
    out = np.empty(len(team), dtype="<U1")
    for h in np.unique(d["period"]):
        idx = np.flatnonzero(d["period"] == h)
        ch, ca = (np.r_[0, np.cumsum(team[idx] == s)] for s in "HA")
        lo, hi = np.maximum(0, np.arange(len(idx)) - VOTE_W), np.minimum(len(idx), np.arange(len(idx)) + VOTE_W + 1)
        h_, a_ = ch[hi] - ch[lo], ca[hi] - ca[lo]
        out[idx] = np.where(h_ + a_ == 0, "D", np.where(h_ >= a_, "H", "A"))
    return out

## 3. Load the training data

In [9]:
t0 = time.time()
data = {p.stem: load(p) for p in sorted(TRAIN_DIR.glob("J03W*.parquet")) + sorted(MET_DIR.glob("MET*.parquet"))}
MATCHES = list(data)
METS = [m for m in MATCHES if m.startswith("MET")]
print(f"{len(MATCHES)} matches ({len(METS)} Metrica), {sum(len(d['y']) for d in data.values()):,} frames in {time.time() - t0:.0f} s")
print("Playing direction of H per half:", {m: d["directions"] for m, d in data.items()})

10 matches (3 Metrica), 1,427,530 frames in 310 s
Playing direction of H per half: {'J03WMX': (-1.0, 1.0), 'J03WN1': (-1.0, 1.0), 'J03WOH': (-1.0, 1.0), 'J03WOY': (-1.0, 1.0), 'J03WPY': (-1.0, 1.0), 'J03WQQ': (-1.0, 1.0), 'J03WR9': (1.0, -1.0), 'MET1': (1.0, -1.0), 'MET2': (-1.0, 1.0), 'MET3': (-1.0, 1.0)}


## 4. Choose the switching costs (leave-one-match-out)

For each of the 10 matches the three models are fitted on the other 9. We choose the cost pair with the highest mean spellF1 on the **Metrica matches**, because they are closest to the challenge. DFL serves as a control. Because the choice uses the same out-of-fold predictions, the Metrica score is slightly optimistic.

In [10]:
t0 = time.time()
oof = {}
for m in MATCHES:
    rest = [x for x in MATCHES if x != m]
    oof[m] = (p_d(fit_d(rest), data[m]), p_h(fit_h(rest), data[m]), fit_restart(rest))

cost_scores = {c: pd.Series({m: score(data[m], decode(data[m], *oof[m][:2], *c, oof[m][2]))["spell_f1"] for m in MATCHES}) for c in COST_GRID}
C_HA, C_D = max(COST_GRID, key=lambda c: cost_scores[c][METS].mean())
print(f"Chosen: c_HA = {C_HA}, c_D = {C_D} · {time.time() - t0:.0f} s")

pd.DataFrame({f"c_HA={c[0]}, c_D={c[1]}": {"Metrica (criterion)": s[METS].mean(), "DFL (control)": s.drop(METS).mean()}
              for c, s in cost_scores.items()}).T.round(4)

Chosen: c_HA = 10, c_D = 40 · 350 s


,Metrica (criterion),DFL (control)
"c_HA=5, c_D=20",0.6421,0.4723
"c_HA=5, c_D=40",0.6445,0.4729
"c_HA=5, c_D=80",0.6410,0.4725
"c_HA=5, c_D=160",0.6321,0.4683
"c_HA=10, c_D=20",0.6438,0.4765
"c_HA=10, c_D=40",0.6463,0.4770
"c_HA=10, c_D=80",0.6427,0.4765
"c_HA=10, c_D=160",0.6291,0.4717
"c_HA=20, c_D=20",0.6318,0.4775
"c_HA=20, c_D=40",0.6342,0.4781


In [11]:
final_oof = pd.DataFrame({m: score(data[m], decode(data[m], *oof[m][:2], C_HA, C_D, oof[m][2])) for m in MATCHES}).T
base_oof = pd.DataFrame({m: score(data[m], baseline(data[m])) for m in MATCHES}).T
is_met = final_oof.index.isin(METS)
pd.DataFrame({("Metrica", "Baseline smooth50"): base_oof[is_met].mean(), ("Metrica", "our approach"): final_oof[is_met].mean(),
              ("DFL", "Baseline smooth50"): base_oof[~is_met].mean(), ("DFL", "our approach"): final_oof[~is_met].mean()}).T.round(3)

spell_f1  macro_f1  alive_dead_f1  n_pred_spells  \
Metrica Baseline smooth50     0.405     0.813          0.914        501.000   
        our approach          0.646     0.917          0.970        255.333   
DFL     Baseline smooth50     0.214     0.522          0.677        441.000   
        our approach          0.477     0.874          0.972        274.429   

                           n_gt_spells  
Metrica Baseline smooth50      372.333  
        our approach           372.333  
DFL     Baseline smooth50      437.286  
        our approach           437.286

## 5. Final models on all 10 matches

The models are fitted once on all matches and not changed afterwards. Parameters and feature lists are stored in `frozen_model.json` for traceability.

In [12]:
model_d, model_h, model_rs = fit_d(MATCHES), fit_h(MATCHES), fit_restart(MATCHES)
frozen = {
    "training_data": ["DFL/IDSSE open data (figshare 28196177, CC BY 4.0), 7 matches, degradation scenario d67s",
                      "Metrica Sports sample data (github.com/metrica-sports/sample-data, commit e706dd5), 3 matches, event-derived labels (notebook 03), not degraded"],
    "selection": "costs chosen by mean out-of-fold spellF1 on the 3 Metrica matches (leave-one-match-out over 10 matches)",
    "parameters": {"c_HA": int(C_HA), "c_D": int(C_D), "w_restart": W_RS, "snap": SNAP, "max_ball_speed": MAX_BALL_SPEED,
                   "max_player_speed": MAX_PLAYER_SPEED, "subsample": SUBSAMPLE},
    "models": {
        "dead": {"type": "HistGradientBoostingClassifier", "params": GBM_PARAMS, "features": D_FEATURES, "n_iter": int(model_d.n_iter_)},
        "home_given_live": {"type": "HistGradientBoostingClassifier", "params": GBM_PARAMS, "features": H_FEATURES, "n_iter": int(model_h.n_iter_)},
        "restart_team": {"type": "LogisticRegression", "features": RS_FEATURES,
                         "coef_standardized": dict(zip(RS_FEATURES, map(float, model_rs[-1].coef_[0][:len(RS_FEATURES)])))},
    },
}
(OUT / "frozen_model.json").write_text(json.dumps(frozen, indent=1, ensure_ascii=False))
print(f"D model: {len(D_FEATURES)} features · H model: {len(H_FEATURES)} features · restart model: {len(RS_FEATURES)} features")

D model: 159 features · H model: 59 features · restart model: 4 features


## 6. Predict the challenge match

Same loader as in notebook 02, then plausibility rule, features and decoder. Only the graded halves are predicted; all other frames get `X`.

In [13]:
meta = ET.parse(ATD_META).getroot()
spec = meta.find("./DataFormatSpecifications/DataFormatSpecification")
team_side = {t.get("id"): s for t, s in zip(meta.findall("./Metadata/Teams/Team"), "HA")}
player_team = {p.get("id"): team_side[p.get("teamId")] for p in meta.findall("./Metadata/Players/Player")}
channel_player = {c.get("id"): c.get("playerId") for c in meta.findall("./Metadata/PlayerChannels/PlayerChannel")}
side_c = np.array([player_team[channel_player[g.find("PlayerChannelRef").get("playerChannelId")]] for g in spec.findall("./SplitRegister/SplitRegister")])
print("Teams:", [t.findtext("Name") for t in meta.findall("./Metadata/Teams/Team")], "→ H, A")

xy = np.empty((N_FRAMES, len(side_c) + 1, 2), dtype=np.float32)
with ATD_RAW.open() as fh:
    for i, line in enumerate(fh):
        _, pl, bl = line.strip().split(":")
        xy[i] = np.fromstring(pl.replace(";", ",") + "," + bl, sep=",").reshape(-1, 3)[:, :2]
xy = (xy - 0.5) * PITCH_CH

graded, period = np.zeros(N_FRAMES, bool), np.zeros(N_FRAMES, int)
for k, (a, b) in enumerate(HALVES, 1):
    graded[a:b + 1], period[a:b + 1] = True, k
ball_c, players_c = clean(xy[graded, -1], xy[graded, :-1])
ch = features({"ball": ball_c, "players": players_c, "period": period[graded], "side": side_c}, tuple(PITCH_CH / 2))
print("Playing direction of H per half (challenge):", ch["directions"])

labels = np.full(N_FRAMES, "X", dtype="<U1")
labels[graded] = decode(ch, p_d(model_d, ch), p_h(model_h, ch), C_HA, C_D, model_rs)
pd.Series(labels).value_counts().rename("frames").to_frame().T

Teams: ['Barcelona', 'Atletic Club'] → H, A


Playing direction of H per half (challenge): (1.0, -1.0)


,D,H,X,A
frames,69111,54536,40462,35210


## 7. Write, validate and hash the file

In [14]:
pd.DataFrame({"frame": np.arange(N_FRAMES), "label": labels}).to_csv(SUBMISSION, index=False, lineterminator="\n")
check = pm.validate_submission(pm.read_label_csv(SUBMISSION), N_FRAMES)
sha = hashlib.sha256(SUBMISSION.read_bytes()).hexdigest()
print("Validation:", "OK" if check is None else f"ERROR: {check}")
print(f"File: {SUBMISSION.relative_to(ROOT)} · {SUBMISSION.stat().st_size:,} bytes · SHA-256 {sha}")

pd.Series(pm.possession_stats(pm.label_codes(labels), [(a, b + 1) for a, b in HALVES])).rename(
    {"n_spells": "possessions", "mean_duration_s": "mean duration [s]", "home_share": "H share"}).to_frame("prediction").T.round(3)

Validation: OK
File: output/HSA_Connection.csv · 1,682,773 bytes · SHA-256 92a943257cf73aaa057aa81cf29902732b72f2b545b95410ae5b2608b1a0f8fb


,possessions,mean duration [s],H share
prediction,305.0,11.77,0.608


## Result

- On the external matches (leave-one-match-out) the approach reaches clearly higher spellF1 than the official baseline (table in section 4).
- The submission file is validated. Its SHA-256 must be `92a943257cf73aaa057aa81cf29902732b72f2b545b95410ae5b2608b1a0f8fb`; then it is bit-identical to the submitted version.
- Neither the challenge ground truth nor the video was used in any step.